# Partition pipeline – every partition on its own, one combined result

Cabinets and workers belong to a **partition**: the team whose work it is. `TRIS_1` is the internal ground team;
`TROS_1`, `TROS_2`, ... are external teams. Every partition is planned on its own:

* its cabinets are clustered only with each other, with the rules of its row in `parameters.csv`;
* its people available (`TOTAL_PEOPLE`) are its own workers, unless its row gives a number;
* its clusters are staffed only by its own workers, and its secondary cabinets come only from its own cabinets.

For each partition this notebook runs `ClusteringModel.ipynb` and then `WorkerLayer.ipynb` (a copy with the files and
the partition filled in; the notebooks themselves are not changed), and then it combines the results.

**A new partition** only needs its name in the `partition` column of both files and a row in `parameters.csv`.
**The same cabinet in two partitions** is fine: it is planned in each of them. In the combined tables a cabinet is
identified by `partition` and `CabinetId` together.

**Writes** to `OUTPUT_FOLDER`:

| Path | What is in it |
|---|---|
| `<partition>/` | that partition's clustering results, map and executed notebook |
| `<partition>/workers/` | that partition's Worker Layer results, map and executed notebook |
| `combined/worker_cabinets.csv` | the main output: `staff_id`, `CabinetId`, `type` (`Primary`/`Secondary`), `partition`, `cluster_id` |
| `combined/` | every other table of all partitions together, and `summary.csv` |

Cluster IDs get the partition in front (`TRIS_1-C001`), so they are unique across partitions.

## Setup and parameters
Paths are relative to the project root; the model rules are in `PARAMETER_FILE`. Start the road server
(`start_osrm.bat`) first.

In [ ]:
CABINET_FILE = "Dataset/mock_cabinets.csv"     # cabinets: cabinetid, cabinetlat, cabinetlong, demand, partition
WORKER_FILE = "Dataset/mock_workers.csv"       # workers: staff_id, node, latitude, longitude, partition
PARAMETER_FILE = "parameters.csv"              # the model rules, one row per partition
OUTPUT_FOLDER = "outputs/partitions"           # one folder per partition, and combined/
PARTITIONS = None                              # only these, e.g. ["TROS_1"] (None = every partition in CABINET_FILE)
CACHE_FILE = "outputs/cache/osrm_cache.json"   # saved road distances, shared by every partition
TIMEOUT_S = 4 * 3600                           # per notebook cell (the fit cell runs the model several times)

In [ ]:
# Install any missing library into the Python that is running this notebook (e.g. Anaconda),
# so "Run All" works on a fresh machine. Already-installed libraries are left alone.
import importlib.util
import subprocess
import sys

REQUIRED = {"pandas": "pandas", "numpy": "numpy", "nbformat": "nbformat", "nbclient": "nbclient",
            "ipykernel": "ipykernel"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing libraries:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("Python", sys.version.split()[0], "- all required libraries available")

In [ ]:
import json
import re
import shutil
import time
from datetime import datetime
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

ROOT = Path.cwd()   # the project root
OUT_DIR = ROOT / OUTPUT_FOLDER
COMBINED = OUT_DIR / "combined"
COMBINED.mkdir(parents=True, exist_ok=True)
START = time.time()
pd.set_option("display.width", 140)

## Check the partitions
Before anything runs: one row per partition with its cabinets, real demand, workers, `TOTAL_PEOPLE` and what they can
take per day. The run stops here if a partition has no row in `parameters.csv`, has no workers and no `TOTAL_PEOPLE`,
lists a cabinet twice, or if a cabinet or worker has no partition. Workers of a partition without cabinets are spare.

In [ ]:
def column(df, name, file):
    # The column called `name` in any upper/lower case; stops if the file has none.
    found = next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)
    if found is None:
        raise ValueError(f"{file} has no column {name}")
    return found


def partitions(df, file):
    # Every row's partition (stripped); stops if one is empty.
    names = df[column(df, "partition", file)].fillna("").str.strip()
    if (names == "").any():
        raise ValueError(f"{file}: {int((names == '').sum())} row(s) have no partition")
    return names


cab = pd.read_csv(ROOT / CABINET_FILE, dtype=str)
cab["partition"] = partitions(cab, CABINET_FILE)
cab["CabinetId"] = cab[column(cab, "cabinetid", CABINET_FILE)]
cab["demand"] = pd.to_numeric(cab[column(cab, "demand", CABINET_FILE)], errors="coerce")
wk = pd.read_csv(ROOT / WORKER_FILE, dtype=str)
wk["partition"] = partitions(wk, WORKER_FILE)
wk["staff_id"] = wk[column(wk, "staff_id", WORKER_FILE)].fillna("").str.strip()
rules = pd.read_csv(ROOT / PARAMETER_FILE, dtype=str).fillna("")
rules.index = rules[column(rules, "partition", PARAMETER_FILE)].str.strip()

found = sorted(cab["partition"].unique())
TODO = found if PARTITIONS is None else list(PARTITIONS)
unknown = [p for p in TODO if p not in found]
if unknown:
    raise ValueError(f"PARTITIONS: {CABINET_FILE} has no cabinets in {', '.join(unknown)} (it has {', '.join(found)})")

problems = []
twice = cab[cab.duplicated(["partition", "CabinetId"], keep=False)]
if len(twice):
    problems.append("cabinet(s) twice in the same partition: " + ", ".join(
        f"{c} ({p})" for p, c in twice[["partition", "CabinetId"]].drop_duplicates().head(10).to_numpy()))
overview = []
for p in TODO:
    here, crew = cab[cab["partition"] == p], wk[wk["partition"] == p]
    row = rules.loc[p] if p in rules.index else None
    if row is None:
        problems.append(f"{p} has no row in {PARAMETER_FILE}")
    given = "" if row is None else str(row[column(rules, "TOTAL_PEOPLE", PARAMETER_FILE)]).strip()
    people = int(float(given)) if given else len(crew)
    if row is not None and people == 0:
        problems.append(f"{p} has no workers in {WORKER_FILE} and no TOTAL_PEOPLE in {PARAMETER_FILE}")
    per_person = None if row is None else float(row[column(rules, "DEMAND_PER_PERSON", PARAMETER_FILE)] or "nan")
    overview.append({"partition": p, "cabinets": len(here), "real_demand_per_day": here["demand"].sum(),
                     "workers": len(crew), "TOTAL_PEOPLE": people,
                     "TOTAL_PEOPLE_from": "parameters.csv" if given else "workers",
                     "DEMAND_PER_PERSON": per_person,
                     "capacity_per_day": None if per_person is None else people * per_person})
if problems:
    raise ValueError("Cannot start:\n- " + "\n- ".join(problems))

NO_CABINETS = sorted(set(wk["partition"]) - set(found))
if NO_CABINETS:
    print(f"Workers in partitions without cabinets (they will be spare): "
          + ", ".join(f"{p} ({int((wk['partition'] == p).sum())})" for p in NO_CABINETS))
shared = cab.groupby("CabinetId")["partition"].nunique()
print(f"{len(cab)} cabinets in {len(found)} partitions, {len(wk)} workers; running {len(TODO)}: {', '.join(TODO)}")
print(f"{int((shared > 1).sum())} cabinet(s) are in more than one partition (planned in each of them)")
OVERVIEW = pd.DataFrame(overview)
OVERVIEW.round(2)

## Run every partition
One partition after another, sharing the saved road distances in `CACHE_FILE`: first the clustering, into
`OUTPUT_FOLDER/<partition>`, then the Worker Layer, into `<partition>/workers/`. A partition without workers skips
the Worker Layer, so all its places stay empty. The executed copies are saved next to the results; if one fails, the
run stops and names the partition.

In [ ]:
def patched(notebook, values):
    # An in-memory copy of a notebook with these values in its parameters cell (the file itself is not changed), the
    # same way dashboard/runner.py does it.
    nb = nbformat.read(ROOT / notebook, as_version=4)
    cell = next(c for c in nb.cells if c.cell_type == "code")
    for name, value in values.items():
        cell.source, n = re.subn(rf"^{name}\s*=.*$", lambda _: f"{name} = {value!r}", cell.source, flags=re.M)
        if n != 1:
            raise RuntimeError(f"parameter {name} not found in {notebook}")
    return nb


def execute(nb, saved_as):
    # Run a notebook copy from the project folder and save it, with its outputs, as saved_as. Returns the error or None.
    client = NotebookClient(nb, timeout=TIMEOUT_S, kernel_name="python3", resources={"metadata": {"path": str(ROOT)}})
    error = None
    try:
        client.execute()
    except CellExecutionError as err:
        error = f"{getattr(err, 'ename', 'Error')}: {getattr(err, 'evalue', str(err))}"
    finally:
        saved_as.parent.mkdir(parents=True, exist_ok=True)
        nbformat.write(nb, saved_as)
    return error


for p in TODO:
    folder, started = f"{OUTPUT_FOLDER}/{p}", time.time()
    print(f"{p}: clustering ...", flush=True)
    copy = patched("ClusteringModel.ipynb", {"DATA_FILE": CABINET_FILE, "OUTPUT_FOLDER": folder,
                                             "PARAMETER_FILE": PARAMETER_FILE, "PARTITION": p,
                                             "WORKER_FILE": WORKER_FILE, "CACHE_FILE": CACHE_FILE})
    error = execute(copy, ROOT / folder / "executed_clustering.ipynb")
    if error:
        raise RuntimeError(f"partition {p}: ClusteringModel.ipynb failed - {error}\n"
                           f"See {folder}/executed_clustering.ipynb")
    if (wk["partition"] == p).any():
        print(f"{p}: workers ...", flush=True)
        copy = patched("WorkerLayer.ipynb", {"RESULTS_FOLDER": folder, "WORKER_FILE": WORKER_FILE, "PARTITION": p,
                                             "OUTPUT_FOLDER": f"{folder}/workers"})
        error = execute(copy, ROOT / folder / "workers" / "executed_workers.ipynb")
        if error:
            raise RuntimeError(f"partition {p}: WorkerLayer.ipynb failed - {error}\n"
                               f"See {folder}/workers/executed_workers.ipynb")
    else:
        print(f"{p}: no workers in {WORKER_FILE}, Worker Layer skipped (all places stay empty)")
    print(f"{p}: done in {time.time() - started:.0f} s", flush=True)

## Combine the partitions
Every partition's tables are put together with a `partition` column, and every cluster ID gets its partition in
front. In `combined/`:

* `worker_cabinets.csv`: the main output. `Primary` = a cabinet of the worker's own cluster, `Secondary` = backup
  work in another cluster of the same partition.
* `worker_assignments.csv` (including the workers of partitions without cabinets), `cluster_workers.csv`,
  `clusters.csv`, `teams.csv` (a total per partition and overall), `assignments.csv`, `secondaries.csv`.
* `summary.csv`: per partition and in total, with the rules and the demand scale used.
* `parameters_used.csv` and `pipeline.json`: the rules and the files of this run (the dashboard reads them).

In [ ]:
IDS = {"CabinetId": str, "cluster_id": str, "primary_cluster_id": str, "staff_id": str, "node": str,
       "staff_ids": str, "nodes": str, "partition": str}


def read(path):
    return pd.read_csv(path, dtype=IDS, keep_default_na=False, na_values=[""])


def tagged(df, p, ids=("cluster_id",)):
    # The table with a partition column in front and the partition in front of every cluster ID.
    df = df.copy()
    for c in ids:
        df[c] = [f"{p}-{x}" if isinstance(x, str) and x else "" for x in df[c]]
    df.insert(0, "partition", p)
    return df


def no_workers_table(clusters):
    # cluster_workers for a partition without workers: every place empty.
    out = clusters[["partition", "cluster_id", "people"]].copy()
    out["assigned"], out["missing"] = 0, out["people"]
    out["staff_ids"] = out["nodes"] = ""
    out[["centre_lat", "centre_long"]] = clusters[["centre_lat", "centre_long"]]
    return out


parts = {name: [] for name in ["clusters", "teams", "assignments", "secondaries", "worker_assignments",
                               "cluster_workers", "worker_cabinets", "rules"]}
for p in TODO:
    folder = OUT_DIR / p
    clusters = tagged(read(folder / "clusters.csv"), p)
    parts["clusters"].append(clusters)
    teams = read(folder / "teams.csv")
    teams = tagged(teams[teams["cluster_id"] != "Total"], p)
    teams["team"] = [f"{c} : {n} {'Person' if n == 1 else 'People'}" for c, n in zip(teams["cluster_id"], teams["people"])]
    parts["teams"].append(teams)
    parts["assignments"].append(tagged(read(folder / "assignments.csv"), p))
    parts["secondaries"].append(tagged(read(folder / "secondaries.csv"), p, ("cluster_id", "primary_cluster_id")))
    parts["rules"].append(read(folder / "rules.csv"))
    if (folder / "workers" / "worker_assignments.csv").exists() and (wk["partition"] == p).any():
        parts["worker_assignments"].append(tagged(read(folder / "workers" / "worker_assignments.csv"), p))
        parts["cluster_workers"].append(tagged(read(folder / "workers" / "cluster_workers.csv"), p))
        parts["worker_cabinets"].append(tagged(read(folder / "workers" / "worker_cabinets.csv"), p))
    else:
        parts["cluster_workers"].append(no_workers_table(clusters))
if PARTITIONS is None:   # workers of partitions without cabinets: spare
    for p in NO_CABINETS:
        crew = wk[wk["partition"] == p]
        parts["worker_assignments"].append(pd.DataFrame({
            "partition": p, "staff_id": crew["staff_id"], "node": crew[column(wk, "node", WORKER_FILE)],
            "cluster_id": "", "latitude": pd.to_numeric(crew[column(wk, "latitude", WORKER_FILE)]),
            "longitude": pd.to_numeric(crew[column(wk, "longitude", WORKER_FILE)]), "status": "spare"}))

combined = {name: pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
            for name, tables in parts.items()}
wc = combined["worker_cabinets"]
combined["worker_cabinets"] = (wc[["staff_id", "CabinetId", "type", "partition", "cluster_id"]] if len(wc)
                               else pd.DataFrame(columns=["staff_id", "CabinetId", "type", "partition", "cluster_id"]))

# a Total row per partition and one for everything in teams.csv
teams = combined["teams"]
totals = [pd.DataFrame([{"partition": p, "cluster_id": "Total", "team": f"{int(t['people'].sum())} People per day",
                         "people": int(t["people"].sum()), "demand_per_day": t["demand_per_day"].sum(),
                         "demand_rule": ""}]) for p, t in teams.groupby("partition", sort=False)]
totals.append(pd.DataFrame([{"partition": "Total", "cluster_id": "Total",
                             "team": f"{int(teams['people'].sum())} People per day",
                             "people": int(teams["people"].sum()), "demand_per_day": teams["demand_per_day"].sum(),
                             "demand_rule": ""}]))
combined["teams"] = pd.concat([teams, *totals], ignore_index=True)

# summary: one row per partition, then the total
wa, cw = combined["worker_assignments"], combined["cluster_workers"]
rows = []
for p in TODO + (NO_CABINETS if PARTITIONS is None else []):
    cl = combined["clusters"][combined["clusters"]["partition"] == p]
    w = wa[wa["partition"] == p] if len(wa) else wa
    rule = combined["rules"][combined["rules"]["partition"] == p]
    rows.append({"partition": p, "cabinets": int(cl["n_cabinets"].sum()),
                 "real_demand_per_day": cl["real_demand_per_day"].sum(), "clusters": len(cl),
                 "people_needed": int(cl["people"].sum()), "workers": int((wk["partition"] == p).sum()),
                 "workers_placed": int((w["status"] == "assigned").sum()) if len(w) else 0,
                 "spare": int((w["status"] == "spare").sum()) if len(w) else 0,
                 "empty_places": int(cw.loc[cw["partition"] == p, "missing"].sum()),
                 **(rule.drop(columns="partition").iloc[0].to_dict() if len(rule) else {})})
summary = pd.DataFrame(rows)
counts = ["cabinets", "real_demand_per_day", "clusters", "people_needed", "workers", "workers_placed", "spare",
          "empty_places"]
summary = pd.concat([summary, pd.DataFrame([{"partition": "Total", **summary[counts].sum().to_dict()}])],
                    ignore_index=True)
for c in summary.columns[1:]:   # whole numbers stay whole (the Total row has no rules)
    values = pd.to_numeric(summary[c], errors="coerce").dropna()
    if c != "real_demand_per_day" and len(values) and (values == values.round()).all():
        summary[c] = pd.to_numeric(summary[c], errors="coerce").astype("Int64")
combined["summary"] = summary

for name in ["worker_cabinets", "worker_assignments", "cluster_workers", "clusters", "teams", "assignments",
             "secondaries", "summary"]:
    combined[name].round(4).to_csv(COMBINED / f"{name}.csv", index=False)
shutil.copyfile(ROOT / PARAMETER_FILE, COMBINED / "parameters_used.csv")
(COMBINED / "pipeline.json").write_text(json.dumps({   # what this run was, for the dashboard
    "created": f"{datetime.now():%Y-%m-%d %H:%M}", "CABINET_FILE": CABINET_FILE, "WORKER_FILE": WORKER_FILE,
    "PARAMETER_FILE": PARAMETER_FILE, "CACHE_FILE": CACHE_FILE, "partitions": TODO}, indent=1), encoding="utf-8")
print(f"saved {len(combined['worker_cabinets'])} worker-cabinet rows "
      f"({int((combined['worker_cabinets']['type'] == 'Primary').sum())} primary) and the combined tables in "
      f"{OUTPUT_FOLDER}/combined ({time.time() - START:.0f} s so far)")

## Validation
The notebook stops if any of these fails:

1. Every cabinet (`partition` + `CabinetId`) is in exactly one cluster of its own partition.
2. Every worker appears once, in their own partition.
3. A worker's cluster and all their cabinets belong to the worker's partition, and every `Primary` cabinet is in the
   worker's cluster.
4. Every secondary cabinet belongs to the partition of the cluster that lists it.

In [ ]:
def key(df):
    # (partition, CabinetId) pairs, as one string each
    return df["partition"] + "|" + df["CabinetId"]


def validate(combined):
    a, s, w, wcab = (combined[n] for n in ["assignments", "secondaries", "worker_assignments", "worker_cabinets"])
    # 1. every cabinet of the partitions that ran is in exactly one cluster of its own partition
    want = key(cab[cab["partition"].isin(TODO)])
    got = key(a)
    assert got.is_unique, "a cabinet is in more than one cluster of its partition"
    assert set(got) == set(want), f"{len(set(want) ^ set(got))} cabinet(s) missing or unknown in assignments.csv"
    assert (a["cluster_id"].str.rsplit("-", n=1).str[0] == a["partition"]).all(), "a cluster ID has the wrong partition"
    # 2. every worker once, with their own partition
    own = wk.set_index("staff_id")["partition"]
    if len(w):
        assert w["staff_id"].is_unique, "a worker appears twice"
        assert (w["partition"] == w["staff_id"].map(own)).all(), "a worker is in another partition's results"
        placed = w[w["status"] == "assigned"]
        assert (placed["cluster_id"].str.rsplit("-", n=1).str[0] == placed["partition"]).all(), \
            "a worker is in a cluster of another partition"
    # 3. a worker's cabinets: own partition, Primary = own cluster
    if len(wcab):
        assert (wcab["partition"] == wcab["staff_id"].map(own)).all(), "a worker has cabinets of another partition"
        assert key(wcab).isin(set(got)).all(), "worker_cabinets.csv lists a cabinet that is not in its partition"
        cluster_of = dict(zip(key(a), a["cluster_id"]))
        prim = wcab[wcab["type"] == "Primary"]
        assert (key(prim).map(cluster_of) == prim["cluster_id"]).all(), "a Primary cabinet is not in the worker's cluster"
        assert not wcab.duplicated(["staff_id", "partition", "CabinetId"]).any(), "a cabinet is listed twice for a worker"
    # 4. secondaries come from the same partition
    if len(s):
        assert key(s).isin(set(got)).all(), "a secondary cabinet is not a cabinet of the cluster's partition"
        assert (s["primary_cluster_id"].str.rsplit("-", n=1).str[0] == s["partition"]).all(), \
            "a secondary cabinet belongs to another partition"
    print(f"all checks passed: {len(TODO)} partition(s), {len(a)} cabinets, {len(w)} workers "
          f"({int((w['status'] == 'assigned').sum()) if len(w) else 0} placed), {len(wcab)} worker-cabinet rows")


validate(combined)

### Result summary
One row per partition and the total. `TOTAL_PEOPLE` is the number planned with; `DEMAND_SCALE` is the share of the
real demand kept to fit them (above 1 = scaled up).

In [ ]:
print(f"Run time: {time.time() - START:.0f} s")
summary.round(2)